# Week 4 — Web App, Final Evaluation & Project Report

**Goal:** Deploy the model as a Flask web app, run comprehensive evaluation, and summarise the internship.

### Deliverables
1. Full BLEU score comparison table (LSTM vs Transformer)
2. Error analysis & failure modes
3. Flask web app demo
4. Final project report

In [ ]:
import os, sys
sys.path.insert(0, os.path.dirname(os.getcwd()))

import torch
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from PIL import Image
import torchvision.transforms as T

import config
from src.vocabulary import Vocabulary
from src.dataset import load_captions_df, build_dataloaders
from src.lstm_model import CaptioningLSTM
from src.transformer_model import CaptioningTransformer
from src.evaluate import compute_bleu
from src.inference import beam_search

print('Device:', config.DEVICE)

## 1. Load Both Models

In [ ]:
df    = load_captions_df(config.CAPTIONS_FILE)
vocab = Vocabulary.load(os.path.join(config.MODELS_DIR, 'vocab.pkl'))
_, _, test_loader = build_dataloaders(df, vocab, config.IMAGES_DIR)

lstm_model = CaptioningLSTM(vocab_size=len(vocab))
lstm_model.load_state_dict(
    torch.load(os.path.join(config.MODELS_DIR, 'lstm_best.pt'), map_location=config.DEVICE)
)
lstm_model.to(config.DEVICE).eval()

tf_model = CaptioningTransformer(vocab_size=len(vocab))
tf_model.load_state_dict(
    torch.load(os.path.join(config.MODELS_DIR, 'transformer_best.pt'), map_location=config.DEVICE)
)
tf_model.to(config.DEVICE).eval()

print('Both models loaded ✓')

## 2. BLEU Score Comparison

In [ ]:
print('=== LSTM ===')
bleu_lstm = compute_bleu(lstm_model, test_loader, vocab, device=config.DEVICE)

print('=== Transformer ===')
bleu_tf = compute_bleu(tf_model, test_loader, vocab, device=config.DEVICE)

# Summary table
results = pd.DataFrame({
    'Model'  : ['CNN + LSTM', 'CNN + Transformer'],
    'BLEU-1' : [bleu_lstm['bleu1'], bleu_tf['bleu1']],
    'BLEU-2' : [bleu_lstm['bleu2'], bleu_tf['bleu2']],
    'BLEU-3' : [bleu_lstm['bleu3'], bleu_tf['bleu3']],
    'BLEU-4' : [bleu_lstm['bleu4'], bleu_tf['bleu4']],
})
print('\n', results.to_string(index=False))

# Bar chart
x   = np.arange(4)
w   = 0.35
fig, ax = plt.subplots(figsize=(9, 5))
lstm_scores = [bleu_lstm[f'bleu{i}'] for i in range(1,5)]
tf_scores   = [bleu_tf[f'bleu{i}']   for i in range(1,5)]
ax.bar(x - w/2, lstm_scores, w, label='LSTM',        color='#7c6aff')
ax.bar(x + w/2, tf_scores,   w, label='Transformer', color='#ff6b9d')
ax.set_xticks(x); ax.set_xticklabels(['BLEU-1','BLEU-2','BLEU-3','BLEU-4'])
ax.set_ylabel('Score'); ax.set_title('BLEU Score Comparison')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(config.OUTPUTS_DIR, 'bleu_comparison.png'), dpi=120)
plt.show()

## 3. Error Analysis — Failure Cases

In [ ]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

preprocess = T.Compose([
    T.Resize((224,224)), T.ToTensor(),
    T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])
smoother = SmoothingFunction().method1

sample = df.drop_duplicates('image').sample(50, random_state=77)
scores = []
for _, row in sample.iterrows():
    img = Image.open(os.path.join(config.IMAGES_DIR, row['image'])).convert('RGB')
    t   = preprocess(img).unsqueeze(0).to(config.DEVICE)
    with torch.no_grad():
        gen = tf_model.caption(t, vocab)[0]
    ref = row['caption'].lower().split()
    hyp = gen.split()
    sc  = sentence_bleu([ref], hyp, smoothing_function=smoother)
    scores.append((sc, row['image'], row['caption'], gen))

scores.sort(key=lambda x: x[0])
worst = scores[:4]

fig, axes = plt.subplots(1, 4, figsize=(16, 5))
for ax, (sc, img_name, ref, gen) in zip(axes, worst):
    img = Image.open(os.path.join(config.IMAGES_DIR, img_name)).convert('RGB')
    ax.imshow(img)
    ax.set_title(f'BLEU: {sc:.3f}\nGen: {gen}\nRef: {ref[:50]}', fontsize=7)
    ax.axis('off')

plt.suptitle('Worst-Performing Captions (Error Analysis)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(config.OUTPUTS_DIR, 'error_analysis.png'), dpi=120)
plt.show()

## 4. Launch Web App

In [ ]:
print('To launch the web app, run in terminal:')
print('  python app.py')
print('Then open: http://localhost:5000')
print()
print('Or run the cell below to start from notebook (blocks the kernel):')

In [ ]:
# Uncomment to start server inline (opens on port 5000)
# import subprocess
# subprocess.Popen(['python', '../app.py'])
# print('Server started at http://localhost:5000')

## 5. Final Project Summary

In [ ]:
print('='*60)
print('  IMAGE CAPTIONING INTERNSHIP — FINAL SUMMARY')
print('='*60)
print(f'  Dataset  : Flickr8k ({len(df)} image-caption pairs)')
print(f'  Vocab    : {len(vocab)} words')
print(f'  Device   : {config.DEVICE}')
print()
print('  CNN Encoder  : ResNet50 (pretrained on ImageNet)')
print('  LSTM Decoder : 2-layer LSTM, hidden=512, embed=256')
print('  Transformer  : 3-layer, 8 heads, d_model=512')
print()
print('  BLEU-4 Results:')
print(f'    LSTM        : {bleu_lstm["bleu4"]:.4f}')
print(f'    Transformer : {bleu_tf["bleu4"]:.4f}')
print()
print('  Possible improvements:')
print('    • Attention mechanisms over spatial CNN features')
print('    • Use CLIP or ViT as backbone encoder')
print('    • Train on COCO (larger dataset)')
print('    • CIDEr / METEOR / ROUGE evaluation')
print('='*60)
print('  Internship complete ✓')
print('='*60)